# Cross-Dataset Benchmark

Notebook này chạy cross-dataset theo protocol trong paper `2503.14356v1` và workflow IMPROVE CSA.

Protocol:

- Source dataset: dùng train/val từ `{source}_split_{fold}_train.txt` và `{source}_split_{fold}_val.txt`.
- Nếu `source == target`: evaluate trên `{source}_split_{fold}_test.txt`, tức diagonal within-dataset entry `G[source, source]`.
- Nếu `source != target`: evaluate trên toàn bộ `{target}_all.txt`, tức off-diagonal cross-dataset entry `G[source, target]`.
- Lặp qua các folds, rồi lấy mean/std R2 để tạo ma trận `G`.
- Tính thêm `Ga`, `Gn`, `Gna` như paper.

Mặc định notebook để smoke test rất nhỏ. Full run 5 datasets x 5 targets x 10 folds x nhiều model sẽ rất nặng, đặc biệt với GraphDRP và SimpleLinearNN.

In [1]:
from pathlib import Path
import importlib
import sys

ROOT = Path.cwd()
if ROOT.name == "new_notebook":
    ROOT = ROOT.parent

NOTEBOOK_DIR = ROOT / "new_notebook"
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import within_dataset_4models
importlib.reload(within_dataset_4models)

from within_dataset_4models import (
    BENCHMARK_DATASETS,
    make_default_config,
    plot_all_cross_dataset_g_matrices,
    run_cross_dataset_benchmark,
    summarize_cross_dataset_results,
)

cfg = make_default_config(ROOT)
cfg

BenchmarkConfig(root=PosixPath('/Users/vietanh/Desktop/ML-Predicting-drug-response'), datasets=['CCLE'], folds=[0], models=['ridge', 'random_forest', 'lightgbm', 'graphdrp', 'simple_linear_nn'], use_lincs_symbol_genes=True, top_ge_features=512, top_mordred_features=512, max_train_rows=None, max_eval_rows=None, random_forest_epochs=100, random_forest_patience=50, graphdrp_epochs=150, graphdrp_batch_size=256, graphdrp_patience=20, graphdrp_learning_rate=0.0001, simple_nn_epochs=300, simple_nn_batch_size=64, simple_nn_val_batch_size=64, simple_nn_patience=50, simple_nn_learning_rate=0.01, simple_nn_dropout=0.01, simple_nn_model='default', random_state=42)

## Config

In [2]:
# Smoke test: source CCLE -> target gCSI, fold 0, two fast tabular models.
source_datasets = ["CCLE"]
target_datasets = ["gCSI"]
cfg.folds = [0]
cfg.models = ["ridge", "random_forest"]

# Full paper-style cross-dataset matrix. Uncomment when ready.
# source_datasets = BENCHMARK_DATASETS
# target_datasets = BENCHMARK_DATASETS
# cfg.folds = list(range(10))
# cfg.models = ["ridge", "random_forest", "lightgbm", "graphdrp", "simple_linear_nn"]

# For quick debugging only. Set back to None for real results.
cfg.max_train_rows = None
cfg.max_eval_rows = None

# Smaller debug values if you include slower models.
# cfg.random_forest_epochs = 10
# cfg.graphdrp_epochs = 3
# cfg.simple_nn_epochs = 3

only_cross_dataset = False

cfg, source_datasets, target_datasets

(BenchmarkConfig(root=PosixPath('/Users/vietanh/Desktop/ML-Predicting-drug-response'), datasets=['CCLE'], folds=[0], models=['ridge', 'random_forest'], use_lincs_symbol_genes=True, top_ge_features=512, top_mordred_features=512, max_train_rows=None, max_eval_rows=None, random_forest_epochs=100, random_forest_patience=50, graphdrp_epochs=150, graphdrp_batch_size=256, graphdrp_patience=20, graphdrp_learning_rate=0.0001, simple_nn_epochs=300, simple_nn_batch_size=64, simple_nn_val_batch_size=64, simple_nn_patience=50, simple_nn_learning_rate=0.01, simple_nn_dropout=0.01, simple_nn_model='default', random_state=42),
 ['CCLE'],
 ['gCSI'])

## Run Cross-Dataset Benchmark

In [3]:
results = run_cross_dataset_benchmark(
    cfg,
    source_datasets=source_datasets,
    target_datasets=target_datasets,
    only_cross_dataset=only_cross_dataset,
)
display(results)

Source=CCLE | fold=0
Target=gCSI | train=7,616, val=952, target=4,941 | tabular features=1,024
  Training ridge
    val: n=952 RMSE=0.0821 MAE=0.0630 R2=0.7174 Pearson=0.8473
    test: n=4,941 RMSE=0.1662 MAE=0.1306 R2=0.2277 Pearson=0.5474
  Training random_forest
    val: n=952 RMSE=0.0796 MAE=0.0621 R2=0.7346 Pearson=0.8572
    test: n=4,941 RMSE=0.2143 MAE=0.1632 R2=-0.2842 Pearson=0.2624
Saved: /Users/vietanh/Desktop/ML-Predicting-drug-response/new_notebook/results/cross_dataset_results.csv


,analysis,dataset,fold,stage,model,train_seconds,n_train,n_features,status,n,...,mae,r2,pearson,source_dataset,target_dataset,target_split_file,preprocess,split_train_rows,split_val_rows,split_target_rows
0,cross_dataset,CCLE,0,val,ridge,0.530626,7616,1024,ok,952,...,0.062970,0.717354,0.847328,CCLE,CCLE,CCLE_split_0_val.txt,tabular gene expression + Mordred drug descrip...,7616,952,4941
1,cross_dataset,CCLE,0,test,ridge,0.530626,7616,1024,ok,4941,...,0.130606,0.227744,0.547380,CCLE,gCSI,gCSI_all.txt,tabular gene expression + Mordred drug descrip...,7616,952,4941
2,cross_dataset,CCLE,0,val,random_forest,164.802859,7616,1024,ok,952,...,0.062060,0.734584,0.857179,CCLE,CCLE,CCLE_split_0_val.txt,official-style tabular gene expression + Mordr...,7616,952,4941
3,cross_dataset,CCLE,0,test,random_forest,164.802859,7616,1024,ok,4941,...,0.163199,-0.284195,0.262428,CCLE,gCSI,gCSI_all.txt,official-style tabular gene expression + Mordr...,7616,952,4941


## Summary: G, Ga, Gn, Gna

In [4]:
summary, aggregates = summarize_cross_dataset_results(results, cfg.out_dir)
display(summary)
display(aggregates)

Saved: /Users/vietanh/Desktop/ML-Predicting-drug-response/new_notebook/results/cross_dataset_summary.csv
Saved: /Users/vietanh/Desktop/ML-Predicting-drug-response/new_notebook/results/cross_dataset_aggregates_Ga_Gna.csv


,model,source_dataset,target_dataset,folds,n_test_mean,r2_mean,r2_std,rmse_mean,rmse_std,mae_mean,pearson_mean,train_seconds_mean
0,random_forest,CCLE,gCSI,1,4941.0,-0.284195,NaN,0.214318,NaN,0.163199,0.262428,164.802859
1,ridge,CCLE,gCSI,1,4941.0,0.227744,NaN,0.166197,NaN,0.130606,0.547380,0.530626


,model,source_dataset,Ga_mean_cross_r2,Gna_mean_normalized_cross_r2,within_r2,n_cross_targets
0,random_forest,CCLE,-0.284195,-0.284195,NaN,1
1,ridge,CCLE,0.227744,0.227744,NaN,1


## G Matrix Heatmaps

Vẽ ma trận `G[source, target]` kiểu paper. Số lớn là mean R2, số trong ngoặc là std across folds nếu có nhiều fold.

In [5]:
from IPython.display import display

figures = plot_all_cross_dataset_g_matrices(
    summary,
    dataset_order=BENCHMARK_DATASETS,
    out_dir=cfg.out_dir,
)

for fig, _ax in figures.values():
    display(fig)


Saved: /Users/vietanh/Desktop/ML-Predicting-drug-response/new_notebook/results/cross_dataset_G_matrix_random_forest.png
Saved: /Users/vietanh/Desktop/ML-Predicting-drug-response/new_notebook/results/cross_dataset_G_matrix_ridge.png


<Figure size 1400x1160 with 2 Axes>

<Figure size 1400x1160 with 2 Axes>

## Inspect R2 Matrix Files

For each model, summary writes:

- `cross_dataset_G_r2_mean_{model}.csv`
- `cross_dataset_G_r2_std_{model}.csv`
- `cross_dataset_Gn_r2_{model}.csv`
- `cross_dataset_aggregates_Ga_Gna.csv`

In [6]:
print("Output dir:", cfg.out_dir.resolve())
for path in sorted(cfg.out_dir.glob("cross_dataset_*.csv")):
    print(path.name)

Output dir: /Users/vietanh/Desktop/ML-Predicting-drug-response/new_notebook/results
cross_dataset_G_r2_mean_random_forest.csv
cross_dataset_G_r2_mean_ridge.csv
cross_dataset_G_r2_std_random_forest.csv
cross_dataset_G_r2_std_ridge.csv
cross_dataset_Gn_r2_random_forest.csv
cross_dataset_Gn_r2_ridge.csv
cross_dataset_aggregates_Ga_Gna.csv
cross_dataset_results.csv
cross_dataset_summary.csv
